# 07 · V-optimal item selection (MIRT)

Same protocol as 05, but with the D chosen in 06 and a selection rule that works in D dimensions.

- **Selectors:** random, MIRT anchors, static V-optimal, adaptive V-optimal
- **Estimators:** accuracy, p-IRT, gp-IRT

In D dimensions an item's information is a matrix, $w_i\alpha_i\alpha_i^\top$ with $w_i=p_i(1-p_i)$,
so there is no single number to rank by. Instead we pick the item that most reduces the average
predicted-logit variance over all items, $\mathrm{tr}(MC)$, where $C$ is the posterior covariance
of θ and $M=\frac{1}{Q}\sum_i\alpha_i\alpha_i^\top$. Adding item $i$ reduces it by

$$g_i=\frac{w_i\,(C\alpha_i)^\top M(C\alpha_i)}{1+w_i\alpha_i^\top C\alpha_i}$$

([Krumdick et al., Eq. 6](https://arxiv.org/html/2604.01418v1#S3.SS2)).
**Static** uses the training-average $w_i$; **adaptive** refits θ after each answer.
Both start from the training prior and never repeat an item.
$g_i$ is a local approximation for choosing items, not a calibrated score interval.

In [ ]:
import sys

sys.path.insert(0, "../scripts/irt")

import numpy as np
import pandas as pd
from diagnostics import paired_comparison
from efficbench import ROOT, load
from protocol import fit_context, make_splits
from run_progression import VOPTIMAL_SELECTORS, run_selection

from cabench.plotting import plot_score_errors

ks = [10, 20, 50, 100, 200, 500]
seeds = range(20)
out = ROOT / "results/irt/progression/voptimal"

Y, models, items = load()
Y = Y.astype(np.float32)
splits = make_splits(Y)
subjects, model_ids, item_ids = items.subject.values, models.model_id.values, items.item_id.values

# Optional: one frozen fitted model per split, to skip fitting.
contexts = None
n_dims = None  # None reads the D chosen in 06

## Run

The static and adaptive designs are deterministic, so every seed gets the same items.
Seeds only matter for pairing with the random and anchor runs.

In [ ]:
if contexts is None:
    if n_dims is None:
        n_dims = int(pd.read_csv(out.parent / "mirt/chosen_dimension.csv").n_dims.iloc[0])
    contexts = [fit_context(Y[train], subjects, n_dims=n_dims) for train, _, _ in splits]

metrics, records = run_selection(
    Y, splits, contexts, VOPTIMAL_SELECTORS, out, model_ids, ks=ks, seeds=seeds, item_ids=item_ids
)

## Error by selector

Test MAE in percentage points, ±1 SD across splits.

In [ ]:
plot_score_errors(records);

In [ ]:
100 * metrics.groupby(["k", "selector", "estimator"]).mae.mean().unstack("estimator")

## Paired changes

Each gp-IRT method vs random, then adaptive vs static V-optimal. Negative ΔMAE means lower error than the reference. The CI bootstraps the five split means (seeds averaged first). The splits overlap, so read it as sensitivity to the split, not as five independent samples.

In [ ]:
vs_random = paired_comparison(records)
vs_static = paired_comparison(records, baseline="voptimal_static + gp-IRT")
paired = pd.concat(
    [
        vs_random[vs_random.method.str.endswith("+ gp-IRT")],
        vs_static[vs_static.method == "voptimal_adaptive + gp-IRT"],
    ],
    ignore_index=True,
)
paired[["delta_mae", "ci_low", "ci_high"]] *= 100
paired[["baseline", "method", "k", "delta_mae", "ci_low", "ci_high", "splits_won", "n_splits"]]

This is where the single-benchmark scoring work stops. Cost-aware selection,
cross-benchmark prediction and checkpoint change detection are left for later.